# ConfluxDuck Colab demo
This Colab-ready notebook demonstrates the typical ConfluxDuck workflow in Google Colab: clone the repo, install dependencies, mount Google Drive for persistence, load CSVs into a Drive-backed DuckDB, infer relationships, and export to Google Sheets.

Run the cells in order. Persist files (unified.duckdb, relationships.json) to Drive so they survive session restarts.


In [ ]:
# 1) Clone the repo and install runtime dependencies
!git clone https://github.com/wrbryan/conflux-duck.git || true
%cd conflux-duck
!pip install -r requirements.txt --quiet

In [ ]:
# 2) Mount Google Drive and set workspace paths
from google.colab import drive
drive.mount('/content/drive')
import os
WORKDIR = '/content/drive/MyDrive/conflux-duck-work'
os.makedirs(WORKDIR, exist_ok=True)
DB_PATH = f"{WORKDIR}/unified.duckdb"
REL_PATH = f"{WORKDIR}/relationships.json"
print('DB_PATH =', DB_PATH)


## 3) Load CSVs into a Drive-backed DuckDB
This will create (or overwrite) the named tables in the DB. The example repo includes a small `data/transactions.csv` you can use to follow along.

**Note (Updated):** The `load_csvs.py` script requires `json.dumps()` for proper JSON serialization of column schemas. If you encounter a `ConversionException` about malformed JSON, refer to the debugging cells below.


In [ ]:
!python scripts/load_csvs.py --data-dir data --out-db "$DB_PATH"

### Troubleshooting: Fix JSON serialization in load_csvs.py
If the above cell fails with `Conversion Error: Malformed JSON`, run these cells to patch the script (Updated - Gemini debug session).


In [ ]:
# Patch 1: Replace str(s["columns"]) with json.dumps(s["columns"])
import json
import os

script_path = 'scripts/load_csvs.py'

with open(script_path, 'r') as f:
    script_content = f.read()

old_string = 'con.execute("INSERT INTO __table_schemas VALUES(?, ?)", [s["table"], str(s["columns"])])'
new_string = 'con.execute("INSERT INTO __table_schemas VALUES(?, ?)", [s["table"], json.dumps(s["columns"])])'

if old_string in script_content:
    modified_content = script_content.replace(old_string, new_string)
    with open(script_path, 'w') as f:
        f.write(modified_content)
    print(f"✓ Patched '{script_path}' to use json.dumps for column serialization.")
else:
    print(f"Note: Could not find exact string. Script may already be patched.")


In [ ]:
# Patch 2: Add 'import json' to the imports section
import os

script_path = 'scripts/load_csvs.py'

with open(script_path, 'r') as f:
    script_content = f.read()

if 'import json' not in script_content:
    old_import_string = 'import re'
    new_import_string = 'import re\nimport json'
    
    if old_import_string in script_content:
        modified_content = script_content.replace(old_import_string, new_import_string, 1)
        with open(script_path, 'w') as f:
            f.write(modified_content)
        print(f"✓ Added 'import json' to '{script_path}'.")
    else:
        print(f"Could not find import anchor. Prepending import json.")
        modified_content = 'import json\n' + script_content
        with open(script_path, 'w') as f:
            f.write(modified_content)
        print(f"✓ Added 'import json' to the beginning.")
else:
    print(f"✓ 'import json' already present in '{script_path}'.")


In [ ]:
# Verify the patch and re-run the loader
print("=== Current load_csvs.py imports ===")
!head -15 scripts/load_csvs.py
print("\n=== Re-running load_csvs.py ===")
!python scripts/load_csvs.py --data-dir data --out-db "$DB_PATH"

## 4) (Optional) Merge other DuckDB files into the unified DB
If you have other .duckdb files on Drive, attach them and copy their tables into the unified DB.


In [ ]:
# Example: if you have another DB at /content/drive/MyDrive/other.duckdb, merge it:
# !python scripts/merge_duckdbs.py --out-db "$DB_PATH" --attach /content/drive/MyDrive/other.duckdb

## 5) Run relationship inference
This scans the DB and writes suggestions to relationships.json.

**Note (Updated):** Relationship inference requires at least two tables in the database. If only one table is loaded, the output will be an empty list.


In [ ]:
!python scripts/infer_relationships.py --db "$DB_PATH" --out "$REL_PATH"

## 6) Interactive review of inferred relationships
Try the ipywidgets-based interactive reviewer first. If widgets are not available or do not render properly in Colab, use the fallback DataFrame-based manual selection shown next.


In [ ]:
# Interactive reviewer (may or may not work in Colab depending on the runtime).
try:
    from scripts.review_relationships import review_relationships
    accepted = review_relationships(REL_PATH, db=DB_PATH)
    print('Accepted (in-memory):', len(accepted))
except Exception as e:
    print('Interactive reviewer failed or widgets unavailable:', e)
    print('Run the fallback cell to review suggestions as a DataFrame and accept indices manually.')


## 7) Fallback review (DataFrame + manual accept)
If the interactive UI doesn't work, this cell displays the suggested relationships as a DataFrame; after inspecting it, edit `accepted_indices` to the row indices you want to accept and run the cell to write them to the database.


In [ ]:
import json, pandas as pd, duckdb
sugg = json.load(open(REL_PATH))
df = pd.json_normalize(sugg)
df.index.name = 'index'
display(df)
# After inspecting, set accepted_indices to the indices you want to accept (e.g. [0,2])
accepted_indices = []  # << EDIT this list after reviewing
if accepted_indices:
    accepted = df.loc[accepted_indices].to_dict(orient='records')
    con = duckdb.connect(DB_PATH)
    con.execute("CREATE TABLE IF NOT EXISTS __relationships(from_table VARCHAR, from_col VARCHAR, to_table VARCHAR, to_col VARCHAR, reason VARCHAR, score DOUBLE)")
    for r in accepted:
        con.execute("INSERT INTO __relationships VALUES(?, ?, ?, ?, ?, ?)",
                    [r.get('from_table') or r.get('table'), r.get('from_col') or r.get('column'),
                     r.get('to_table'), r.get('to_col'), r.get('reason'), r.get('score')])
    con.close()
    print('Wrote accepted relationships to DB')
else:
    print('No indices selected; no changes written.')


## 8) Quick reporting & charts
If a `transactions` table exists, produce a small category summary and a bar chart using Plotly.


In [ ]:
import duckdb, plotly.express as px
con = duckdb.connect(DB_PATH)
# Check for transactions table
has_transactions = con.execute("SELECT count(*) FROM information_schema.tables WHERE table_name='transactions'").fetchone()[0] > 0
if has_transactions:
    df = con.execute("SELECT category, -SUM(CASE WHEN amount<0 THEN amount ELSE 0 END) AS expenses FROM transactions GROUP BY category").df()
    display(df)
    fig = px.bar(df, x='category', y='expenses', title='Expenses by category')
    fig.show()
else:
    print('No transactions table in DB. Run the loader first.')


## 9) Export a query result to Google Sheets (user OAuth)
This example uses Colab's user OAuth flow to avoid storing service-account keys in the notebook. Run the cell and follow the auth prompt.


In [ ]:
# Install lightweight Google client libs if needed
!pip install --quiet gspread google-auth
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
# Replace with your query
try:
    df = con.execute("SELECT * FROM transactions LIMIT 100").df()
    sh = gc.create('ConfluxDuck Transactions (Colab)')
    worksheet = sh.get_worksheet(0)
    worksheet.update([df.columns.values.tolist()] + df.values.tolist())
    print('Sheet URL:', sh.url)
except Exception as e:
    print('Export failed or no transactions table:', e)


---
## Notes
- Persist DB and relationships.json to Drive.
- ipywidgets may not behave consistently in Colab; use the DataFrame fallback when necessary.
- Avoid storing service-account keys in the repo; load them from Drive only if strictly necessary.

## Updates (Gemini Debug Session)
- **Section 3:** Added troubleshooting note for JSON serialization issues.
- **Cells 3.1-3.3:** Added debug cells to patch `load_csvs.py` with `json.dumps()` and add `import json`.
- **Section 5:** Added note that relationship inference requires multiple tables.
